# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. We will examine its record sets and fields, then perform exploratory data analysis—all referencing fields by their `@id` as required by the Croissant standard.

### Dataset Source
The dataset schema and metadata are provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id`s.

We'll enumerate all record sets, then enumerate the fields (and columns where relevant) for each, showing their names and IDs.

In [ ]:
# List all record sets and their fields using `@id`
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets detected via mlcroissant - likely this dataset defines records at top-level.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '')}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {getattr(field, 'name', '')}: @id={getattr(field, 'id', '')}, type={getattr(field, 'data_type', '')}")
        print()
if not record_sets:
    # Attempt to load records with no explicit record_set
    print("\nTrying to list top-level records (record_set_id=None):")
    try:
        for i, record in enumerate(dataset.records(record_set=None)):
            if i>2:
                break
            print(record)
    except Exception as e:
        print(f"Could not list records: {e}")

## 3. Data Extraction

Load data from specific record set(s) into pandas DataFrames for further analysis. All entities are referenced by their `@id` field as per the Croissant specification.

The following code will automatically attempt to extract data from all available record sets. If no record sets are present, it will extract from the entire dataset as a single set.

In [ ]:
# Build list of record set @ids
from collections import OrderedDict
dataframes = {}

if not record_sets:
    # No record_sets: treat as a single table
    default_rs_id = None
    print("Extracting all records from dataset (single main table)...")
    records = list(dataset.records(record_set=None))  # all data
    df = pd.DataFrame(records)
    dataframes["main"] = df
    print(f"Loaded {len(df)} records.")
    print("First 3 field @ids (columns):", df.columns.values[:3])
    display_columns = df.columns.tolist()
    df.head()
else:
    record_set_ids = [rs.id for rs in record_sets]
    print("Extracting data from all record sets by their @id:")
    for rs in record_sets:
        rs_id = rs.id
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  - '{rs.name}' (@id={rs_id}): {len(df)} records, columns: {df.columns.values[:3]}")
    # Show columns for first record set as example
    display_columns = list(dataframes[record_set_ids[0]].columns)
    print("\nFirst few columns in the first record set:", display_columns)
    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)

We now perform typical EDA tasks. First, choose a numeric field using its full `@id` (not just its label). We'll apply filtering, normalization, and group-by aggregations referencing Croissant field `@id`s.

In [ ]:
# --- Choose the DataFrame and field `@id`s for EDA ---

# Use 'main' if no record sets exist; else pick the first record set
if not record_sets:
    df = dataframes['main']
    record_set_id = 'main'
else:
    record_set_id = record_sets[0].id
    df = dataframes[record_set_id]

print(f"\nAvailable field @ids (columns) in this table:\n", list(df.columns))

# Let's try to infer a numeric field by scanning the DataFrame dtypes
numeric_candidates = df.select_dtypes(include=['int', 'float']).columns.tolist()
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    # If numeric field is not detected automatically, prompt user
    print("No obvious numeric field found; please update 'numeric_field_id' manually.")
    numeric_field_id = df.columns[0]

# Let's also try to infer a categorical/group field
group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field_id = None
for col in group_candidates:
    # Exclude fields that appear to be IDs
    if not col.endswith('@id') and 'id' not in col.lower():
        group_field_id = col
        break
if group_field_id is not None:
    print(f"Grouping by field: {group_field_id}")

# --- EDA: Filtering, Normalization, Grouping ---
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records where field '@id' {numeric_field_id} > {threshold:.2f}:")
display_cols = [numeric_field_id] + ([group_field_id] if group_field_id else [])
print(filtered_df[display_cols].head())

if len(filtered_df) > 0:
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}' values:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())
else:
    print("No records after filtering. Cannot normalize.")

# Grouped aggregation example
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    grouped_df = grouped_df.reset_index().rename(columns={numeric_field_id: f"{numeric_field_id}_mean"})
    print(f"\nMean of '{numeric_field_id}' grouped by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field and (optionally) compare across a group. All axes/fields are referenced by Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

plt.figure(figsize=(10,5))
sns.histplot(df[numeric_field_id], kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id is not None and group_field_id in df:
    plt.figure(figsize=(12,6))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded and described the FAIR^2 dataset using its Croissant schema and `mlcroissant`.
- Explored all available record sets and fields, referencing them by Croissant `@id` as best practice.
- Loaded records into pandas DataFrames, filtered and normalized a numeric field, and performed group-wise aggregation.
- Visualized key dataset features for overview and further analysis.

For publication or further analysis, always reference Croissant fields by their `@id` field for maximum reproducibility and FAIR provenance.